# TensorRT-LLM

NVIDIA's high-performance inference library optimized for Large Language Models.

## Table of Contents

1. [Introduction](#introduction)
2. [Key Features](#key-features)
3. [Architecture Overview](#architecture)
4. [Installation](#installation)
5. [Basic Usage](#basic-usage)
6. [Advanced Features](#advanced-features)
7. [Use Cases](#use-cases)
8. [Best Practices](#best-practices)
9. [Common Pitfalls](#pitfalls)
10. [Performance Optimization](#performance)
11. [Production Deployment](#deployment)
12. [Monitoring and Observability](#monitoring)
13. [Troubleshooting](#troubleshooting)
14. [Comparison with Alternatives](#comparison)
15. [Resources](#resources)

## Introduction

**TensorRT-LLM** is NVIDIA's flagship library for achieving maximum performance when serving Large Language Models on NVIDIA GPUs.

### What is it?

TensorRT-LLM is a high-performance inference toolkit that:
- Optimizes LLM inference using TensorRT engines
- Provides custom CUDA kernels for transformer operations
- Supports advanced quantization (INT4, INT8, FP8)
- Enables multi-GPU tensor and pipeline parallelism
- Delivers peak performance on NVIDIA GPUs

### Why use it?

Key benefits:
- **Maximum Performance**: 2-8x faster than PyTorch baseline
- **Memory Efficiency**: INT4 quantization reduces memory by 8x
- **Advanced Batching**: In-flight batching for high throughput
- **Multi-GPU**: Efficient scaling to 8+ GPUs for 70B+ models
- **Production Ready**: Powers NVIDIA AI Enterprise
- **Wide Model Support**: Llama, GPT, Mistral, Falcon, StarCoder, etc.

### When to use it?

TensorRT-LLM is ideal when:
- You need **absolute peak performance** on NVIDIA GPUs
- Serving latency-sensitive applications (sub-100ms p99)
- Running large models (70B+) across multiple GPUs
- Cost optimization through aggressive quantization
- Building on NVIDIA infrastructure (DGX, HGX)

**Trade-off**: Requires model compilation (build time) but delivers unmatched runtime performance.

## Key Features

### Core Capabilities of TensorRT-LLM

| Feature | Description | Benefit |
|---------|-------------|----------|
| **Optimized Kernels** | Hand-tuned CUDA kernels for attention, MLP, LayerNorm | 2-8x faster than PyTorch |
| **INT4/INT8/FP8 Quantization** | Advanced quantization with minimal accuracy loss | 4-8x memory reduction |
| **Multi-GPU Parallelism** | Tensor and pipeline parallelism | Serve 70B+ models |
| **In-Flight Batching** | Continuous batching with iteration-level scheduling | Maximum GPU utilization |
| **Flash Attention** | Memory-efficient attention implementation | Longer context support |
| **KV Cache Management** | Efficient key-value cache with paging | Reduced memory overhead |
| **Custom Plugins** | Extensible plugin system for custom ops | Flexibility for research |
| **NVIDIA Triton Integration** | Native Triton backend | Enterprise serving |
| **Multi-Precision** | FP16, BF16, FP8 support | Hardware-specific optimization |
| **AWQ/GPTQ Support** | State-of-the-art quantization methods | Best quality at low precision |

## Architecture Overview

TensorRT-LLM uses a **build-then-serve** architecture:

```
┌─────────────────────────────────────────────────────────┐
│               BUILD PHASE (Offline)                     │
│                                                         │
│  ┌──────────────┐      ┌──────────────┐              │
│  │ Source Model │─────▶│  TensorRT-   │              │
│  │ (HF/GGUF)    │      │  LLM Builder │              │
│  └──────────────┘      └──────┬───────┘              │
│                               │                        │
│                               ▼                        │
│  ┌────────────────────────────────────────┐          │
│  │  Optimization Pass                     │          │
│  │  • Quantization (INT4/INT8/FP8)        │          │
│  │  • Kernel fusion                       │          │
│  │  • Memory layout optimization          │          │
│  │  • Multi-GPU partitioning              │          │
│  └────────────────┬───────────────────────┘          │
│                   │                                    │
│                   ▼                                    │
│  ┌────────────────────────────────────────┐          │
│  │  TensorRT Engine (.engine file)        │          │
│  │  • Optimized for target GPU            │          │
│  │  • Serialized CUDA kernels             │          │
│  └────────────────────────────────────────┘          │
└─────────────────────────────────────────────────────────┘

┌─────────────────────────────────────────────────────────┐
│               RUNTIME PHASE (Online)                    │
│                                                         │
│  ┌──────────────┐                                      │
│  │   Client     │                                      │
│  │   Requests   │                                      │
│  └──────┬───────┘                                      │
│         │                                               │
│         ▼                                               │
│  ┌────────────────────────────────┐                   │
│  │  Batch Manager                 │                   │
│  │  • In-flight batching          │                   │
│  │  • KV cache management         │                   │
│  │  • Request scheduling          │                   │
│  └────────────┬───────────────────┘                   │
│               │                                         │
│               ▼                                         │
│  ┌────────────────────────────────┐                   │
│  │  TensorRT Runtime              │                   │
│  │  • Execute optimized kernels   │                   │
│  │  • Manage GPU memory           │                   │
│  └────────────┬───────────────────┘                   │
│               │                                         │
│               ▼                                         │
│  ┌────────────────────────────────┐                   │
│  │  GPU (Multi-GPU if needed)     │                   │
│  │  • Tensor parallelism          │                   │
│  │  • Pipeline parallelism        │                   │
│  └────────────────────────────────┘                   │
└─────────────────────────────────────────────────────────┘
```

### Key Concepts

1. **Build Phase**: Models are compiled into optimized TensorRT engines (one-time cost)
2. **Engine File**: Serialized, GPU-specific optimized model (not portable across GPU types)
3. **Runtime Phase**: Load engine and serve with minimal overhead
4. **In-Flight Batching**: Requests enter/exit batches dynamically at iteration level

## Installation

### Prerequisites

- **NVIDIA GPU**: Ampere (A100, A10) or newer (H100, L40S)
- **CUDA**: 12.2 or newer
- **Docker**: Highly recommended
- **NVIDIA Container Toolkit**: For GPU access in Docker
- **Python**: 3.10+

### Installation via Docker (Recommended)

TensorRT-LLM is best used through official NVIDIA containers:

In [ ]:
# Pull TensorRT-LLM Docker image
# Run in terminal:
# docker pull nvcr.io/nvidia/tensorrt_llm/release:v0.7.1

# Or build from source
# git clone https://github.com/NVIDIA/TensorRT-LLM.git
# cd TensorRT-LLM
# make -C docker release_build

print("TensorRT-LLM is typically used via Docker containers")
print("See: https://nvidia.github.io/TensorRT-LLM/installation.html")

In [ ]:
# If using pip (for development, less recommended)
# pip install tensorrt_llm==0.7.1 --extra-index-url https://pypi.nvidia.com

# Verify installation
import sys
print(f"Python version: {sys.version}")

# Inside TensorRT-LLM container, you can import:
# import tensorrt_llm
# print(f"TensorRT-LLM version: {tensorrt_llm.__version__}")

## Basic Usage

### Two-Step Process: Build → Run

#### Step 1: Build TensorRT Engine

In [ ]:
# Example: Build Llama-2-7B engine
# This is typically done via command line inside the Docker container

build_command = '''
# Clone model from Hugging Face
git lfs install
git clone https://huggingface.co/meta-llama/Llama-2-7b-hf

# Convert to TensorRT-LLM checkpoint format
python convert_checkpoint.py \\
    --model_dir ./Llama-2-7b-hf \\
    --output_dir ./tllm_checkpoint \\
    --dtype float16

# Build TensorRT engine
trtllm-build \\
    --checkpoint_dir ./tllm_checkpoint \\
    --output_dir ./trt_engines/llama-2-7b \\
    --gemm_plugin float16 \\
    --max_batch_size 8 \\
    --max_input_len 2048 \\
    --max_output_len 512
'''

print("Build command (run in TensorRT-LLM container):")
print(build_command)
print("\nBuild time: 5-15 minutes (one-time cost)")
print("Result: Optimized .engine file ready for inference")

#### Step 2: Run Inference with Built Engine

In [ ]:
# Python inference example (inside TensorRT-LLM environment)

# This code runs inside the TensorRT-LLM container
inference_example = '''
import tensorrt_llm
from tensorrt_llm.runtime import ModelRunner

# Load the built engine
runner = ModelRunner.from_dir(
    engine_dir="./trt_engines/llama-2-7b",
    rank=0  # GPU rank for multi-GPU setups
)

# Prepare input
prompts = [
    "What is machine learning?",
    "Explain quantum computing in simple terms."
]

# Generate
outputs = runner.generate(
    prompts,
    max_new_tokens=100,
    temperature=0.7,
    top_p=0.95,
    end_id=2,  # EOS token ID
)

# Print results
for prompt, output in zip(prompts, outputs):
    print(f"Prompt: {prompt}")
    print(f"Output: {output}")
    print("-" * 80)
'''

print("Python inference example:")
print(inference_example)

### Using the C++ Runtime (Highest Performance)

In [ ]:
# For absolute maximum performance, use C++ API
cpp_example = '''
# Run inference using C++ executable
mpirun -n 1 \\
    --allow-run-as-root \\
    python ../run.py \\
    --engine_dir ./trt_engines/llama-2-7b \\
    --max_output_len 100 \\
    --tokenizer_dir ./Llama-2-7b-hf \\
    --input_text "What is the meaning of life?"
'''

print("C++ runtime for maximum throughput:")
print(cpp_example)

## Advanced Features

### 1. INT4/INT8 Quantization

In [ ]:
# Build with INT4 AWQ quantization (4x memory reduction)
awq_build = '''
# First, quantize the model with AWQ
python quantize.py \\
    --model_dir ./Llama-2-7b-hf \\
    --dtype float16 \\
    --qformat int4_awq \\
    --output_dir ./llama-2-7b-awq \\
    --calib_size 32

# Build engine with quantized weights
trtllm-build \\
    --checkpoint_dir ./llama-2-7b-awq \\
    --output_dir ./trt_engines/llama-2-7b-int4 \\
    --gemm_plugin float16 \\
    --max_batch_size 16 \\
    --max_input_len 2048 \\
    --max_output_len 512
'''

print("INT4 AWQ Quantization:")
print(awq_build)
print("\nBenefits:")
print("• Memory: 7B model fits in 4GB VRAM (vs 14GB FP16)")
print("• Speed: 1.5-2x faster inference")
print("• Quality: <1% accuracy degradation")

In [ ]:
# Alternative: SmoothQuant INT8 (better for older GPUs)
smoothquant_build = '''
# Quantize with SmoothQuant
python quantize.py \\
    --model_dir ./Llama-2-7b-hf \\
    --dtype float16 \\
    --qformat int8_sq \\
    --output_dir ./llama-2-7b-sq \\
    --smoothquant 0.5

# Build with INT8 activations and weights
trtllm-build \\
    --checkpoint_dir ./llama-2-7b-sq \\
    --output_dir ./trt_engines/llama-2-7b-int8 \\
    --gemm_plugin float16
'''

print("SmoothQuant INT8:")
print(smoothquant_build)

### 2. Multi-GPU Tensor Parallelism

In [ ]:
# Serve Llama-2-70B across 4 GPUs with tensor parallelism
multi_gpu_build = '''
# Convert checkpoint for 4-way tensor parallelism
python convert_checkpoint.py \\
    --model_dir ./Llama-2-70b-hf \\
    --output_dir ./tllm_checkpoint_4gpu \\
    --dtype float16 \\
    --tp_size 4

# Build engines (one per GPU)
trtllm-build \\
    --checkpoint_dir ./tllm_checkpoint_4gpu \\
    --output_dir ./trt_engines/llama-2-70b-tp4 \\
    --gemm_plugin float16 \\
    --max_batch_size 16 \\
    --workers 4

# Run inference with MPI (4 GPUs)
mpirun -n 4 \\
    --allow-run-as-root \\
    python run.py \\
    --engine_dir ./trt_engines/llama-2-70b-tp4 \\
    --tokenizer_dir ./Llama-2-70b-hf \\
    --input_text "Your prompt here"
'''

print("Multi-GPU Tensor Parallelism:")
print(multi_gpu_build)
print("\nScaling:")
print("• 7B: 1 GPU")
print("• 13B: 1-2 GPUs")
print("• 70B: 4-8 GPUs")
print("• 175B: 8+ GPUs")

### 3. In-Flight Batching for Maximum Throughput

In [ ]:
# Use NVIDIA Triton with TensorRT-LLM backend for in-flight batching
triton_config = '''
# Triton model config (config.pbtxt)
name: "tensorrt_llm"
backend: "tensorrtllm"
max_batch_size: 32

model_transaction_policy {
  decoupled: True
}

dynamic_batching {
  preferred_batch_size: [8, 16, 32]
  max_queue_delay_microseconds: 1000
}

instance_group [
  {
    count: 1
    kind: KIND_GPU
    gpus: [0]
  }
]

parameters: {
  key: "gpt_model_type"
  value: { string_value: "inflight_batching" }
}

parameters: {
  key: "max_tokens_in_paged_kv_cache"
  value: { string_value: "8192" }
}
'''

print("Triton + TensorRT-LLM for in-flight batching:")
print(triton_config)
print("\nIn-flight batching benefits:")
print("• Requests join/leave batches dynamically")
print("• No padding waste (unlike static batching)")
print("• 2-3x higher throughput")

### 4. Custom Plugins for Research

In [ ]:
# TensorRT-LLM supports custom CUDA plugins
custom_plugin_example = '''
# Example: Custom attention plugin
from tensorrt_llm.functional import gpt_attention
from tensorrt_llm.plugin import TRTLLMPlugin

class MyCustomAttention(TRTLLMPlugin):
    def __init__(self, config):
        super().__init__()
        self.config = config
    
    def forward(self, query, key, value, mask):
        # Your custom CUDA kernel here
        return custom_attention_cuda(query, key, value, mask)

# Register plugin
register_plugin("my_custom_attention", MyCustomAttention)

# Use in model build
trtllm-build \\
    --checkpoint_dir ./checkpoint \\
    --output_dir ./engine \\
    --use_custom_plugin my_custom_attention
'''

print("Custom plugin system for research:")
print(custom_plugin_example)

## Use Cases

### Real-World Applications

#### Use Case 1: Low-Latency Chatbot

In [ ]:
# Serve Mistral-7B with <50ms latency
chatbot_setup = '''
# Build optimized engine
trtllm-build \\
    --checkpoint_dir ./mistral-7b-checkpoint \\
    --output_dir ./engines/mistral-7b-fp8 \\
    --dtype float16 \\
    --gemm_plugin fp8 \\
    --max_batch_size 4 \\
    --max_input_len 1024 \\
    --max_output_len 256

# Serve with Triton
docker run --gpus all -p 8000:8000 \\
    -v ./engines:/engines \\
    nvcr.io/nvidia/tritonserver:23.12-trtllm-python-py3 \\
    tritonserver --model-repository=/engines
'''

print("Low-latency chatbot setup:")
print(chatbot_setup)
print("\nResults:")
print("• Latency: p50=35ms, p99=48ms")
print("• Throughput: 200 req/sec on single A100")
print("• Cost: $0.0005 per 1000 tokens")

#### Use Case 2: Code Generation at Scale

In [ ]:
# Serve CodeLlama-34B with high throughput
code_gen_setup = '''
# Build with INT4 AWQ for efficiency
python quantize.py \\
    --model_dir ./CodeLlama-34b-hf \\
    --qformat int4_awq \\
    --output_dir ./codellama-34b-awq

trtllm-build \\
    --checkpoint_dir ./codellama-34b-awq \\
    --output_dir ./engines/codellama-34b-int4 \\
    --max_batch_size 32 \\
    --max_input_len 4096 \\
    --max_output_len 1024

# Result: 34B model fits in single A100 (40GB)
'''

print("Code generation setup:")
print(code_gen_setup)
print("\nPerformance:")
print("• Model size: 34B params in 17GB VRAM")
print("• Throughput: 15,000 tokens/sec")
print("• Context: 4K tokens input")

#### Use Case 3: Multi-Tenant LLM Platform

In [ ]:
# Serve multiple models on same infrastructure
multi_tenant = '''
# Build multiple engines
# Llama-2-7B for general queries
# CodeLlama-7B for code
# Mistral-7B for creative writing

# Triton model repository structure:
models/
├── llama2-7b/
│   ├── 1/
│   │   └── llama2-7b.engine
│   └── config.pbtxt
├── codellama-7b/
│   ├── 1/
│   │   └── codellama-7b.engine
│   └── config.pbtxt
└── mistral-7b/
    ├── 1/
    │   └── mistral-7b.engine
    └── config.pbtxt

# Route requests based on task type
# Use KV cache sharing for memory efficiency
'''

print("Multi-tenant platform:")
print(multi_tenant)
print("\nBenefits:")
print("• 3 models on single 80GB GPU")
print("• Task-specific routing")
print("• Shared infrastructure, lower cost")

## Best Practices

### Recommended Practices for TensorRT-LLM

#### 1. Model Building

- **Version Pin**: Always use specific TensorRT-LLM versions, engines are not cross-compatible
- **GPU Matching**: Build engines on same GPU architecture as deployment (A100 → A100)
- **Max Lengths**: Set `max_input_len` and `max_output_len` to realistic values (affects memory)
- **Batch Size**: `max_batch_size` should match your concurrency needs
- **Quantization**: Use AWQ for best quality at INT4, SmoothQuant for INT8

#### 2. Performance Tuning

- **KV Cache**: Set `max_tokens_in_paged_kv_cache` based on VRAM (typically 8192-16384)
- **Batch Size**: Larger batches = higher throughput, but higher latency
- **Plugins**: Always enable `gpt_attention_plugin` and `gemm_plugin`
- **Multi-GPU**: Use tensor parallelism (TP) for large models, not pipeline parallelism
- **FP8**: H100 GPUs benefit from FP8 (2x faster than FP16)

#### 3. Production Deployment

- **Use Triton**: TensorRT-LLM backend provides production-grade serving
- **Health Checks**: Monitor GPU memory, throughput, latency
- **Request Queuing**: Implement rate limiting to prevent OOM
- **Fallbacks**: Have backup engines or graceful degradation
- **Monitoring**: Track per-request metrics, GPU utilization

#### 4. Memory Management

- Reserve 2-3GB VRAM for CUDA overhead and KV cache
- Use quantization to fit larger models in available VRAM
- Enable KV cache paging for variable-length sequences
- Monitor `max_tokens_in_paged_kv_cache` vs actual usage

#### 5. Multi-GPU Strategies

```python
# Decision matrix:
# 7B model:   1 GPU (no parallelism)
# 13B model:  1-2 GPUs (TP=2 optional)
# 34B model:  2-4 GPUs (TP=2 or TP=4)
# 70B model:  4-8 GPUs (TP=4 or TP=8)
# 175B model: 8+ GPUs (TP=8)
```

## Common Pitfalls

### What to Avoid When Using TensorRT-LLM

#### 1. GPU Architecture Mismatch

**Problem**: Building engine on A100 and deploying on V100

**Symptom**: `Invalid engine` or `TRT version mismatch` errors

**Solution**: Always build on target GPU architecture. Engines are hardware-specific.

#### 2. Insufficient VRAM for KV Cache

**Problem**: Setting `max_batch_size=32` without considering KV cache memory

**Symptom**: OOM errors during inference, not at startup

**Solution**:
```python
# Estimate VRAM needs:
# Model weights + KV cache + overhead
# KV cache ≈ 2 * batch_size * max_seq_len * hidden_size * num_layers * 2 bytes
# For Llama-2-7B: ~0.4GB per batch per 1K tokens
```

#### 3. Not Setting Max Lengths Correctly

**Problem**: Using default `max_input_len=2048` when you only need 512

**Impact**: Wastes 4x memory on KV cache

**Solution**: Set tight bounds on max lengths based on actual use case

#### 4. Skipping Quantization

**Problem**: Running FP16 when INT4 would suffice

**Impact**: 4x memory waste, lower throughput

**Solution**: Always benchmark quantized models first (AWQ has minimal quality loss)

#### 5. Version Mismatches

**Problem**: Building with TensorRT-LLM 0.7.0, serving with 0.7.1

**Symptom**: Cryptic serialization errors

**Solution**: Pin exact versions in Dockerfile, rebuild engines after upgrades

## Performance Optimization

### Achieving Peak Performance

In [ ]:
# Benchmark script
benchmark_code = '''
# Use built-in benchmarking tools
python benchmarks/benchmark.py \\
    --engine_dir ./engines/llama-2-7b \\
    --batch_size 1,4,8,16,32 \\
    --input_len 128,512,1024 \\
    --output_len 128 \\
    --csv results.csv

# Analyze results
import pandas as pd
df = pd.read_csv('results.csv')
print(df.groupby('batch_size')[['latency_ms', 'throughput_tokens_per_sec']].mean())
'''

print("Benchmarking TensorRT-LLM:")
print(benchmark_code)

In [ ]:
# Performance tuning checklist
tuning_checklist = '''
Performance Optimization Checklist:

✓ Enable all plugins:
  - gpt_attention_plugin
  - gemm_plugin
  - lookup_plugin (for embedding tables)

✓ Use appropriate precision:
  - H100: FP8 (best)
  - A100/L40S: INT4 AWQ (best balance)
  - Older GPUs: INT8 SmoothQuant

✓ Optimize batch size:
  - Latency-sensitive: batch_size=1-4
  - Throughput-focused: batch_size=16-32

✓ Enable in-flight batching:
  - Use Triton backend
  - Set gpt_model_type=inflight_batching

✓ Tune KV cache:
  - Start with 50% of available VRAM
  - Monitor actual usage
  - Adjust based on metrics

✓ Multi-GPU scaling:
  - Use TP (tensor parallelism) not PP (pipeline)
  - TP=2/4/8 based on model size
  - Ensure NVLink between GPUs
'''

print(tuning_checklist)

## Production Deployment

### Deploying TensorRT-LLM at Scale

#### Docker Deployment

In [ ]:
# Production Dockerfile
dockerfile = '''
FROM nvcr.io/nvidia/tritonserver:23.12-trtllm-python-py3

# Copy pre-built engines
COPY engines/ /models/

# Install additional dependencies
RUN pip install prometheus-client

# Health check
HEALTHCHECK --interval=30s --timeout=10s --retries=3 \\
  CMD curl -f http://localhost:8000/v2/health/ready || exit 1

# Run Triton
CMD ["tritonserver", \\
     "--model-repository=/models", \\
     "--http-port=8000", \\
     "--grpc-port=8001", \\
     "--metrics-port=8002", \\
     "--log-verbose=1"]
'''

print("Production Dockerfile:")
print(dockerfile)

#### Kubernetes Deployment

In [ ]:
# Kubernetes manifest for TensorRT-LLM + Triton
k8s_manifest = '''
apiVersion: v1
kind: Service
metadata:
  name: tensorrt-llm-service
spec:
  selector:
    app: tensorrt-llm
  ports:
  - name: http
    port: 8000
  - name: grpc
    port: 8001
  - name: metrics
    port: 8002
  type: LoadBalancer
---
apiVersion: apps/v1
kind: Deployment
metadata:
  name: tensorrt-llm
spec:
  replicas: 2
  selector:
    matchLabels:
      app: tensorrt-llm
  template:
    metadata:
      labels:
        app: tensorrt-llm
    spec:
      containers:
      - name: triton
        image: your-registry/tensorrt-llm-triton:latest
        ports:
        - containerPort: 8000
        - containerPort: 8001
        - containerPort: 8002
        resources:
          limits:
            nvidia.com/gpu: 1
          requests:
            nvidia.com/gpu: 1
            memory: "32Gi"
            cpu: "8"
        livenessProbe:
          httpGet:
            path: /v2/health/live
            port: 8000
          initialDelaySeconds: 30
          periodSeconds: 10
        readinessProbe:
          httpGet:
            path: /v2/health/ready
            port: 8000
          initialDelaySeconds: 30
          periodSeconds: 5
      nodeSelector:
        cloud.google.com/gke-accelerator: nvidia-tesla-a100
'''

print("Kubernetes deployment:")
print(k8s_manifest)

## Monitoring and Observability

### Key Metrics to Track

In [ ]:
# Prometheus metrics configuration
monitoring_setup = '''
# Triton exposes metrics on :8002/metrics
# Key metrics to monitor:

# Request metrics
- nv_inference_request_success (counter)
- nv_inference_request_failure (counter)
- nv_inference_request_duration_us (histogram)
- nv_inference_queue_duration_us (histogram)

# Compute metrics
- nv_inference_compute_input_duration_us
- nv_inference_compute_infer_duration_us
- nv_inference_compute_output_duration_us

# GPU metrics
- nv_gpu_utilization (gauge)
- nv_gpu_memory_total_bytes (gauge)
- nv_gpu_memory_used_bytes (gauge)

# Model-specific
- nv_inference_pending_request_count (gauge)
- nv_inference_exec_count (counter)
'''

print("Monitoring setup:")
print(monitoring_setup)

In [ ]:
# Custom monitoring script
custom_monitoring = '''
import requests
import time
from prometheus_client import start_http_server, Gauge, Histogram

# Custom metrics
token_latency = Histogram('token_latency_seconds', 'Time per token')
active_requests = Gauge('active_requests', 'Number of active requests')
kv_cache_usage = Gauge('kv_cache_usage_ratio', 'KV cache utilization')

def monitor_triton():
    """Monitor Triton server metrics."""
    while True:
        # Get metrics from Triton
        response = requests.get('http://localhost:8002/metrics')
        
        # Parse and expose custom metrics
        # ...
        
        time.sleep(10)

if __name__ == '__main__':
    start_http_server(8003)  # Expose on :8003
    monitor_triton()
'''

print("Custom monitoring:")
print(custom_monitoring)

## Troubleshooting

### Common Issues and Solutions

#### Issue 1: Build Fails with "Out of Memory"

**Symptoms**: `trtllm-build` crashes with CUDA OOM during engine building

**Causes**:
- Model too large for available VRAM
- Building on GPU with less memory than needed

**Solutions**:
```bash
# Use quantization during build
trtllm-build --use_weight_only --weight_only_precision int4_awq

# Or build on a larger GPU (build once, deploy anywhere on same arch)

# Or use multi-GPU build
trtllm-build --workers 2 --tp_size 2
```

#### Issue 2: "Invalid engine" Error at Runtime

**Symptoms**: Engine loads but fails immediately with "Invalid engine version"

**Causes**:
- TensorRT version mismatch between build and runtime
- GPU architecture mismatch

**Solutions**:
```bash
# Check TensorRT-LLM version
python -c "import tensorrt_llm; print(tensorrt_llm.__version__)"

# Rebuild with matching version
# Always use same Docker image for build and serve
```

#### Issue 3: Lower Than Expected Performance

**Symptoms**: Inference is slower than benchmarks

**Causes**:
- Plugins not enabled
- Batch size too small
- CPU bottleneck in tokenization

**Solutions**:
```bash
# Verify plugins enabled
trtllm-build --gpt_attention_plugin float16 --gemm_plugin float16

# Increase batch size
# Use in-flight batching via Triton

# Profile to find bottleneck
nsys profile -o profile python run.py
```

#### Issue 4: CUDA Errors During Multi-GPU Inference

**Symptoms**: Random CUDA errors with multi-GPU setups

**Causes**:
- No NVLink between GPUs
- MPI configuration issues

**Solutions**:
```bash
# Verify NVLink topology
nvidia-smi topo -m

# Use correct MPI launcher
mpirun --allow-run-as-root -n 4 python run.py

# Set CUDA visible devices
export CUDA_VISIBLE_DEVICES=0,1,2,3
```

## Comparison with Alternatives

### How TensorRT-LLM Compares

| Feature | TensorRT-LLM | vLLM | TGI | TorchServe |
|---------|--------------|------|-----|------------|
| **Peak Performance** | ⭐⭐⭐⭐⭐ | ⭐⭐⭐⭐ | ⭐⭐⭐⭐ | ⭐⭐⭐ |
| **Ease of Use** | ⭐⭐ | ⭐⭐⭐⭐⭐ | ⭐⭐⭐⭐ | ⭐⭐⭐ |
| **Setup Time** | ⭐⭐ (build required) | ⭐⭐⭐⭐⭐ | ⭐⭐⭐⭐ | ⭐⭐⭐ |
| **Quantization** | ⭐⭐⭐⭐⭐ (INT4/FP8) | ⭐⭐⭐⭐ | ⭐⭐⭐ | ⭐⭐ |
| **Multi-GPU** | ⭐⭐⭐⭐⭐ | ⭐⭐⭐⭐ | ⭐⭐⭐⭐ | ⭐⭐⭐ |
| **Memory Efficiency** | ⭐⭐⭐⭐⭐ | ⭐⭐⭐⭐ | ⭐⭐⭐⭐ | ⭐⭐⭐ |
| **Flexibility** | ⭐⭐⭐ | ⭐⭐⭐⭐ | ⭐⭐⭐ | ⭐⭐⭐⭐⭐ |
| **Hardware Support** | NVIDIA only | NVIDIA + AMD | NVIDIA + CPU | All platforms |

### When to Choose TensorRT-LLM

**Choose TensorRT-LLM when:**
- ✅ Need absolute peak performance on NVIDIA GPUs
- ✅ Latency is critical (sub-100ms requirements)
- ✅ Running on NVIDIA infrastructure (DGX, HGX, EGX)
- ✅ Have DevOps resources for build pipelines
- ✅ Aggressive cost optimization through quantization
- ✅ Integration with NVIDIA ecosystem (Triton, TAO, etc.)

**Choose alternatives when:**
- ❌ Need fast iteration (vLLM is easier to prototype)
- ❌ Non-NVIDIA hardware (use TorchServe)
- ❌ Hugging Face-first workflow (TGI is simpler)
- ❌ Small team, limited DevOps (build complexity)
- ❌ Frequent model updates (rebuild cost)

### Performance Comparison (Llama-2-7B)

```
Benchmark: A100 80GB, batch_size=8, input=128, output=128

TensorRT-LLM (FP16):     8,500 tokens/sec  | 23ms latency
TensorRT-LLM (INT4):    12,000 tokens/sec  | 18ms latency
vLLM (FP16):             6,800 tokens/sec  | 28ms latency
TGI (FP16):              6,200 tokens/sec  | 31ms latency
TorchServe (FP16):       4,100 tokens/sec  | 47ms latency
```

## Resources

### Official Documentation

- **GitHub**: https://github.com/NVIDIA/TensorRT-LLM
- **Documentation**: https://nvidia.github.io/TensorRT-LLM/
- **Installation Guide**: https://nvidia.github.io/TensorRT-LLM/installation.html
- **Examples**: https://github.com/NVIDIA/TensorRT-LLM/tree/main/examples
- **Docker Images**: https://catalog.ngc.nvidia.com/orgs/nvidia/containers/tritonserver

### Tutorials and Guides

- **Quick Start**: https://github.com/NVIDIA/TensorRT-LLM/blob/main/docs/source/quick-start-guide.md
- **Model Support**: https://github.com/NVIDIA/TensorRT-LLM/blob/main/docs/source/supported_models.md
- **Performance Guide**: https://github.com/NVIDIA/TensorRT-LLM/blob/main/docs/source/performance.md
- **Triton Backend**: https://github.com/triton-inference-server/tensorrtllm_backend

### Research Papers

- **TensorRT**: https://developer.nvidia.com/tensorrt
- **Flash Attention**: https://arxiv.org/abs/2205.14135
- **AWQ Quantization**: https://arxiv.org/abs/2306.00978
- **SmoothQuant**: https://arxiv.org/abs/2211.10438

### Community Resources

- **NVIDIA Developer Forums**: https://forums.developer.nvidia.com/
- **GitHub Issues**: https://github.com/NVIDIA/TensorRT-LLM/issues
- **NVIDIA Discord**: https://discord.gg/nvidia

### Related Technologies

- **Triton Inference Server**: https://github.com/triton-inference-server/server
- **NVIDIA NeMo**: https://github.com/NVIDIA/NeMo
- **CUDA**: https://developer.nvidia.com/cuda-toolkit
- **cuDNN**: https://developer.nvidia.com/cudnn